# **Databases**

### Debeaver Database Setup

Create a database connection

Click:

Database → New Database Connection

You'll get a big list of database types.

Since you're familiar with SQL Server 2022 at work, I'd actually suggest choosing:

SQL Server

That way we can build you a little SQL Server practice environment and your experience will translate pretty directly between work and home.

DBeaver will ask for things like:

Host:       ?
Port:       1433
Database:   ?
Username:   ?
Password:   ?

What I'd do in your lab

This gives us another nice little project. We can run SQL Server in a container/VM in your homelab, then use DBeaver on your CachyOS laptop as the management interface.

The architecture would simply be:

DBeaver on your laptop → network → SQL Server in your lab

Then you could practice the same kinds of things you do at work:
CREATE DATABASE LabDB;

CREATE TABLE Servers (
    id INT PRIMARY KEY,
    hostname VARCHAR(50),
    ip_address VARCHAR(15),
    operating_system VARCHAR(50)
);

INSERT INTO Servers
VALUES (1, 'k8s-controller', '10.0.0.248', 'Oracle Linux');
SELECT * FROM Servers;

And DBeaver will give you the graphical table/database explorer and query-results grid you're accustomed to.

We could even make the database about your homelab infrastructure, which would make learning SQL a little more interesting than creating random customers and employees tables.

For now, open DBeaver and get to the New Database Connection screen. We can build the SQL Server side next and connect the two.

### Redis

#### Definitions
SET command: is used to set the string value of a key
    For example, let's set a key named mykey with the value Hello Redis

#### Installation
sudo apt update -y
sudo apt install -y redis-server
which redis-server
sudo service redis-server start

#### CLI
* redis-cli 
    ping << to test connectivity should return pong
    SET mykey "Hello Redis"
    quit

* redis-cli
    GET mykey
    quit

* redis-cli
    ping
    SET mykey myvalue
    GET mykey << should return "myvalue" >>
    SET user:1001 John
    GET user:1001
    exit

* redis-cli
    EXISTS mykey >> Redis will respond with: (integer) 1 // A response of 1 indicates that the key mykey exists in Redis.
    EXISTS nonexistentkey >> Redis will respond with: (integer) 0 // A response of 0 indicates that the key nonexistentkey does not exist in Redis
    exit

* redis-cli
    DEL mykey >> A response of (integer) 1 indicates that one key was successfully deleted
    EXISTS mykey
    DEL nonexistentkey // A response of (integer) 0 indicates that no keys were deleted (because the key didn't exist)
    DEL user:1001 nonexistentkey >> integer 1
    SET mykey myvalue
    SET user:1001 John
    DEL mykey user:1001 >> (integer) 2

* redis-cli
    SET sessionkey sessionvalue EX 15
    GET sessionkey
    TTL sessionkey

    SET tempkey tempvalue
    GET tempkey
    TTL tempkey
    EXPIRE tempkey 10
    TTL tempkey >> Redis will respond with the number of seconds remaining until the key expires
    

# **HomeLab Builds**

### App Installs

sudo pacman -Syy << That forces pacman to redownload all repository databases
sudo pacman -Syu
sudo cachyos-rate-mirrors

sudo pacman -S dbeaver
    dbeaver
    

### Nagios

# **k8s**

### Definitions

#### ConfigMap
stores non-sensitive configuration data for applications
literally the configs for the pod so that way you can customize and update it at will


### Commands

#### cluster management
kubectl cluster-info
kubectl cluster-info dump > clusterinfo
kubectl api-resources
kubectl api-resources | grep -i pod
kubectl config set-context --current --namespace=ckad-prep-contexts

#### kubectl config
kubectl config current-context
kubectl config set-context --current --namespace=default
kubectl config view --minify --output 'jsonpath={..namespace}'; echo
kubectl config view | grep -i namespace

#### kubectl get
kubectl get nodes --request-timeout=5s
kubectl get no -o wide
kubectl get namespaces
kubectl get pods -n kube-system
kubectl get configmap
kubectl get configmap app-settings --namespace default --ignore-not-found
kubectl get jobs
kubectl get job heartbeat-test
kubectl get pods -l job-name=heartbeat-test

#### kubectl create
kubectl create namespace ckad-prep-contexts
kubectl create configmap app-settings --from-literal=mode=prep
kubectl create job --from=cronjob/cluster-heartbeat heartbeat-test

#### kubectl logs
kubectl logs job/heartbeat-test

#### kubectl describe
kubectl describe job heartbeat-test


### CKAD Training

#### Explore kubectl,Contexts, and Namespaces
mkdir -p /home/labex/project
kubectl config current-context
kubectl config set-context --current --namespace=default
kubectl get nodes --request-timeout=5s
kubectl cluster-info
kubectl get namespaces
kubectl config view --minify --output 'jsonpath={..namespace}'; echo
kubectl get pods --namespace kube-system
kubectl create namespace ckad-prep-contexts
kubectl config set-context --current --namespace=ckad-prep-contexts
kubectl config view --minify --output 'jsonpath={..namespace}'; echo
kubectl get pods --namespace kube-system
kubectl create configmap app-settings --from-literal=mode=prep
kubectl get configmap app-settings --namespace ckad-prep-contexts

#### Inspect k8s API Resources

### **Manifest Files**

#### heartbeat-cronjob.yaml

apiVersion: batch/v1
kind: CronJob
metadata:
  name: cluster-heartbeat
spec:
  schedule: "*/5 * * * *" #*/5 meant "every 5 minutes."
  schedule: "0 * * * *"  #0 means "exactly at minute 0" (the start of the hour).
  schedule: "0 8 * * *"


  jobTemplate:
    spec:
      template:
        spec:
          containers:
            - name: heartbeat
              image: busybox:1.37
              command:
                - /bin/sh
                - -c
                - 'echo "Cluster heartbeat: $(date)"'

          restartPolicy: OnFailure

kubectl apply -f heartbeat-cronjob.yaml
kubectl get cronjobs
kubectl create job --from=cronjob/cluster-heartbeat heartbeat-test
kubectl get jobs
kubectl get pods
kubectl delete job heartbeat-test

# **LInux**

### Cronjobs

#### syntax
0 (Minute) → Exactly at minute 0
8 (Hour) → Exactly at hour 8 (8:00 AM)
* (Day of Month) → Every day
* (Month) → Every month
* (Day of Week) → Every day of the week

### Vim Motions

/search >> after it is highligted
:noh

# **Networking**

### Switch Commands

### Initial Setup
en
sh ip int br
sh int status
sh vlan br
sh run
wr
sh start

conf t
    hostname LanSW1
    no ip domain-lookup
    banner motd @Warning! Unauthorized Access is Prohibited@
    service password encryption
    enable secret ccna
    username admin password admin
    username sysadmin privilege 15 password admin123
    ip domain-name MasonIT
    crypto key generate rsa 1024
    ip ssh version 2


    line console 0
        password Passw0rd
        login
        logging synchronous
        do wr
        do sh run

    line vty 0 15
        login local
        login
        exec timeout 0 0
        logging synchronous
        do wr
        do sh run

### Interface Setup
en 
conf t
    int g0/1
        switchport mode access
        switchport access vlan 10
        no shut
        shut

conf t
    int g0/3
        no shut
        end

### Vlans
en
sh vlan br

conf t
    vlan 10
        name lab1
    no vlan 10

int vlan1
    ip address 10.0.0.254 255.255.255.0
    no shut